In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt

from IPython.display import clear_output
from sklearn.metrics import classification_report #accuracy_score, precision_score, recall_score, 
from sklearn.model_selection import train_test_split

In [ ]:
seed = 42
y_col = "Cover_Type"

## 1. データの読み込み

In [ ]:
df_data = pd.read_csv("../data/data.csv")

## 2. 特徴量の数とモデルの精度の実験

### 2.1 学習関数と元となるデータセットの作成

In [ ]:
def train_predict_eval(
    X_train, y_train, 
    X_val, y_val, 
    X_test, y_test,
):
    
    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=y_train.nunique(),
        class_weight="balanced",
        random_state=seed,
        verbose=-1,        # ログを非表示
        n_estimators=1000  # Early Stoppingを使うため大きめの数値を設定
    )

    print("train starts")
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    print("prediction starts")
    y_pred = model.predict(X_test)

    print("pred ends\n")
    # print(f"percentage_split(ratio to all):{percentage_split}[%]\n",
    #         classification_report(y_test, y_pred))
    # print(X_train_splitted.shape)
    # print(y_train_splitted.value_counts())

    return classification_report(y_test, y_pred, output_dict=True)

In [ ]:
X = df_data.drop(y_col, axis=1).copy()
y = df_data[y_col].copy()

In [ ]:
# データセットを指定した割合で分割する

# テストデータを全体の20%にする
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=seed
)

# 検証データを全体の20%にする
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=seed
)

In [ ]:
X_train.shape

### 2.2 数値カラムとbinariカラムを区別せずにランダムサンプリング

In [ ]:

sample_nums = 20 # 使用する特徴量の数を指定 10,20,30,40で行うbaselineが54なので、50は省略する
reports = [] # classification reportの一時的な格納リスト

# カラムをランダム抽出する
# カラムごとにモデルの予測への貢献度が異なることが多いため、
# 抽出したカラムのモデル精度への影響なのか、カラム数がモデル数に影響を与えたのか、
# 厳密には分離できない。そのため、一定数試行した結果の平均をとることで、特定のカラムの貢献度依存度合いの
# 影響を小さくする

# ループの50回で数十分から100分くらいかかるので、ローカルでやる場合は50を仮の上限にする。
for i in range(50):

    clear_output(wait=True) # forループのなかで、現在の回だけを表示する
    print(f"{i}回目")

    # ランダムでカラム名を抽出する
    sampled_col_names = X_train.sample(
        n=sample_nums,
        axis=1,
        random_state=i # seedを毎回変えることで、毎回ランダムなカラムが抽出される
    ).columns.tolist()

    # 抽出したカラム名で学習、予測を行う
    dict_report = train_predict_eval(
        X_train=X_train[sampled_col_names],
        y_train=y_train, 
        X_val=X_val[sampled_col_names],
        y_val=y_val, 
        X_test=X_test[sampled_col_names],
        y_test=y_test
    )

    df_report = pd.DataFrame(dict_report).T
    
    reports.append(df_report)

# 結果のDataFrameの同じ位置のセル同士で平均をとる
df_mean_report = pd.concat(reports).groupby(level=0).mean()

# 平均を取る前の元データも保存しておく
pd.concat(reports).to_csv(
    f"../outputs/col{sample_nums}_raw.csv",
    index=True,
    encoding="utf-8"
)

# 平均の結果も保存しておく
df_mean_report.to_csv(
    f"../outputs/col{sample_nums}_mean.csv",
    index=True,
    encoding="utf-8"
)


### 2.3 数値カラムとbinariカラムを区別して、元々の比率を保ったままランダムサンプリング

In [ ]:
# バイナリ特徴量と、バイナリ以外の特徴量の設定
# 目的変数の数:1
# バイナリ特徴量の数：44
# バイナリ以外の特徴量の数:10
# バイナリ：バイナリ以外≒8:2

list_non_binary = [
    "Elevation", "Aspect", "Slope", "Horizontal_Distance_To_Hydrology", "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways","Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points"
]

# バイナリ：バイナリ以外≒8:2の比率なので、特徴量の数を減らした時のそれぞれのカラム数を定義する
# データセット分割の際に、10*0.2などをすると、値によっては小数点が発生し、intに変換すると、数がずれる可能性があるので、
# 事前にdictで定義しておく
dict_sample_nums ={
    10:{"non_binary":2, "binary":8},
    20:{"non_binary":4, "binary":16},
    30:{"non_binary":6, "binary":24},
    40:{"non_binary":8, "binary":32}
}


In [ ]:

# 特徴量の種類ごとの比率を維持してランダムサンプリング

sample_nums_stratified = 10 # 使用する特徴量の数を指定 10,20,30,40で行うbaselineが54なので、50は省略する
reports_stratified = [] # classification reportの一時的な格納リスト

# カラムをランダム抽出する
# カラムごとにモデルの予測への貢献度が異なることが多いため、
# 抽出したカラムのモデル精度への影響なのか、カラム数がモデル数に影響を与えたのか、
# 厳密には分離できない。そのため、一定数試行した結果の平均をとることで、特定のカラムの貢献度依存度合いの
# 影響を小さくする

for sample_nums_stratified in [10,20,30,40]:
    reports_stratified = [] # classification reportの一時的な格納リスト
    print(sample_nums_stratified)
    # TODO 上記3行はあとで削除

    # ループの50回で数十分から100分くらいかかるので、ローカルでやる場合は50を仮の上限にする。
    for i in range(50):

        clear_output(wait=True) # forループのなかで、現在の回だけを表示する

        print(f"{i}回目")

        # 層化して、ランダムでカラム名を抽出する

        non_binary_col_names = X_train[list_non_binary].sample(
            n=dict_sample_nums[sample_nums_stratified]["non_binary"],
            axis=1,
            random_state=i # ループごとにseedを毎回変えることで、毎回ランダムなカラムが抽出される+range()なので、再現性もある
        ).columns.tolist()

        binary_col_names = X_train.drop(list_non_binary, axis=1).sample(
            n=dict_sample_nums[sample_nums_stratified]["binary"],
            axis=1,
            random_state=i # ループごとにseedを毎回変えることで、毎回ランダムなカラムが抽出される+range()なので、再現性もある
        ).columns.tolist()

        # binaryカラムと非binaryカラムを統合して、学習に使うカラムリストを作成
        feature_cols = non_binary_col_names + binary_col_names

        # 抽出したカラム名で学習、予測を行う
        dict_report_stratified = train_predict_eval(
            X_train=X_train[feature_cols],
            y_train=y_train, 
            X_val=X_val[feature_cols],
            y_val=y_val, 
            X_test=X_test[feature_cols],
            y_test=y_test
        )

        # 転値して、行=クラス名、列=指標 にする
        df_report_stratified = pd.DataFrame(dict_report_stratified).T
        
        reports_stratified.append(df_report_stratified)

    # 結果を格納したDataFrameの同じ位置のセル同士で平均をとる
    df_mean_report_stratified = pd.concat(reports_stratified).groupby(level=0).mean()

    # 平均を取る前の元データも保存しておく
    pd.concat(reports_stratified).to_csv(
        f"../outputs/stratified_col{sample_nums_stratified}_raw.csv",
        index=True, # indexがクラスなので、クラスをのこす
        encoding="utf-8"
    )

    # 平均の結果も保存しておく
    df_mean_report_stratified.to_csv(
        f"../outputs/stratified_col{sample_nums_stratified}_mean.csv",
        index=True, # indexがクラスなので、クラスをのこす
        encoding="utf-8"
    )
        

## 3. 予測結果の確認

### 3.1 classification_report（平均）の確認

In [ ]:
# 平均と、平均を求める前のデータがあるので、信頼区間とかももとめる（50回なので、どれくらい誤差がありそうか）

In [ ]:
# 完全ランダム

for i in [10,20,30,40]:

    df = pd.read_csv(f"../outputs/col{i}_mean.csv")
    x = range(1,8)

    plt.figure(figsize=(6, 4))

    for col in ["precision", "recall", "f1-score"]:
       
        plt.plot(x, df[col][:7], label=col)

    plt.ylim(0,1)
    plt.yticks([i*0.1 for i in range(11)])
    plt.xlabel("cover_type")
    plt.ylabel("percentage")
    plt.title(f"feature nums: {i}")
    plt.grid()
    plt.legend()
    plt.show()



In [ ]:
df_plot.columns[0]

In [ ]:
# binaryとそれ以外のカラムの比率を保ったままランダム

# df =pd.read_csv("../outputs/stratified_col10_mean.csv")

for i in [10,20,30,40]:

    df = pd.read_csv(f"../outputs/stratified_col{i}_mean.csv")
    
    x = range(1,8)

    plt.figure(figsize=(6, 4))

    for col in ["precision", "recall", "f1-score"]:
       
        plt.plot(x, df[col][:7], label=col)

    plt.ylim(0,1)
    plt.yticks([i*0.1 for i in range(11)])
    plt.xlabel("cover_type")
    plt.ylabel("percentage")
    plt.title(f"feature nums: {i}")
    plt.grid()
    plt.legend()
    plt.show()

### 3.2 95%信頼区間の確認

In [ ]:
def calc_95_ci(stratified_flag):

    col_names =  [
        "precision_mean", "precision_std", "precision_ucl", "precision_lcl",
        "recall_mean","recall_std", "recall_ucl", "recall_lcl", 
        "f1-score_mean","f1-score_std" ,"f1-score_ucl", "f1-score_lcl"
    ]

    df_return = pd.DataFrame(columns=col_names)

    for i in [10,20,30,40]:

        if stratified_flag:
            df = pd.read_csv(f"../outputs/stratified_col{i}_raw.csv")
        else:
            df = pd.read_csv(f"../outputs/col{i}_raw.csv")

        # cover_typeごとにforを回す
        for cover_type in range(1,8):

            # 計算結果を一時的に保存
            list_tmp =[]

            for col in ["precision", "recall", "f1-score"]:
                avg = df[col][df[df.columns[0]]==str(cover_type)].mean()
                std = df[col][df[df.columns[0]]==str(cover_type)].std()
                sample_volume = len(df[col][df[df.columns[0]]==str(cover_type)])

                # 95%信頼区間の上限
                ucl = avg + 1.96*(
                        std/(sample_volume**0.5)
                    )

                # 95%信頼区間の下限
                lcl = avg - 1.96*(
                        std/(sample_volume**0.5)
                    )

                list_tmp.extend([avg, std, ucl, lcl])
            
            df_return.loc[f"{i}_{cover_type}", col_names] = list_tmp

    return df_return

In [ ]:
# 完全ランダム

df_95ci = calc_95_ci(
    stratified_flag = False
)

In [ ]:
df_95ci.sort_values(by=["f1-score_std"], ascending=False)

In [ ]:
# binaryとそれ以外のカラムの比率を保ったままランダム

df_95ci_stratified = calc_95_ci(
    stratified_flag = True
)

In [ ]:
df_95ci_stratified.sort_values(by=["f1-score_std"], ascending=False)


## 4. 特徴量３０個固定＋データサイズ変更

In [ ]:
# 特徴量30個だと、全体を見れば一定の精度を保っているが、クラスごとに確認すると、評価指標が悪化しているものがある
# そのため、データサイズの影響を見るのにちょうどいいと仮定し、特徴量３０個を対象に、学習のデータセットを全体の%かに
# 分割して学習結果を確認する

### 4.1 元となるデータセット作成

In [ ]:
X_datasize = df_data.drop(y_col, axis=1).copy()
y_datasize = df_data[y_col].copy()

# テストデータを全体の20%にする
X_train_val, X_test_datasize, y_train_val, y_test_datasize = train_test_split(
    X_datasize, y_datasize, test_size=0.2, stratify=y_datasize, random_state=seed
)

# 検証データを全体の20%にする
X_train_datasize, X_val_datasize, y_train_datasize, y_val_datasize = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=seed
)

In [ ]:
# 学習データの一部を分割する
# 学習データが全体の60％なので、(percentage_split/60)を指定することで、全体のpercentage_splitにする

percentage_split = 50

_, X_train_splitted, _, y_train_splitted = train_test_split(
    X_train_datasize, y_train_datasize,
    test_size=(percentage_split/60), 
    stratify=y_train_datasize, 
    random_state=seed
)

### 4.2 特徴量30個を対象にデータサイズを変えて精度を確認

In [ ]:

sample_nums_stratified = 30 # 特徴量の数は30個に固定
dfs = [] # classification reportの一時的な格納リスト

# ループの50回で数十分から100分くらいかかるので、ローカルでやる場合は50を仮の上限にする。
for i in range(50):

    clear_output(wait=True) # forループのなかで、現在の回だけを表示する

    print(f"{i}回目")

    # 層化して、ランダムでカラム名を抽出する

    non_binary_col_names = X_train_datasize[list_non_binary].sample(
        n=dict_sample_nums[sample_nums_stratified]["non_binary"],
        axis=1,
        random_state=i # ループごとにseedを毎回変えることで、毎回ランダムなカラムが抽出される+range()なので、再現性もある
    ).columns.tolist()

    binary_col_names = X_train_datasize.drop(list_non_binary, axis=1).sample(
        n=dict_sample_nums[sample_nums_stratified]["binary"],
        axis=1,
        random_state=i # ループごとにseedを毎回変えることで、毎回ランダムなカラムが抽出される+range()なので、再現性もある
    ).columns.tolist()

    # binaryカラムと非binaryカラムを統合して、学習に使うカラムリストを作成
    feature_cols = non_binary_col_names + binary_col_names

    # 抽出したカラム名で学習、予測を行う
    dict_report_stratified_datasize = train_predict_eval(
        X_train=X_train_splitted[feature_cols],
        y_train=y_train_splitted, 
        X_val=X_val_datasize[feature_cols],
        y_val=y_val_datasize, 
        X_test=X_test_datasize[feature_cols],
        y_test=y_test_datasize
    )

    # 転値して、行=クラス名、列=指標 にする
    df_report_stratified_datasize = pd.DataFrame(dict_report_stratified_datasize).T
    
    dfs.append(df_report_stratified_datasize)

# 結果を格納したDataFrameの同じ位置のセル同士で平均をとる
df_mean_report_stratified_datasize = pd.concat(dfs).groupby(level=0).mean()

# 平均を取る前の元データも保存しておく
pd.concat(dfs).to_csv(
    f"../outputs/{percentage_split}_stratified_col{sample_nums_stratified}_raw.csv",
    index=True, # indexがクラスなので、クラスをのこす
    encoding="utf-8"
)

# 平均の結果も保存しておく
df_mean_report_stratified_datasize.to_csv(
    f"../outputs/{percentage_split}_stratified_col{sample_nums_stratified}_mean.csv",
    index=True, # indexがクラスなので、クラスをのこす
    encoding="utf-8"
)

In [ ]:
# 夜のうちに回る関数にしちゃう

def tmp_night_calc(percentage_split):
    X_datasize = df_data.drop(y_col, axis=1).copy()
    y_datasize = df_data[y_col].copy()

    # テストデータを全体の20%にする
    X_train_val, X_test_datasize, y_train_val, y_test_datasize = train_test_split(
        X_datasize, y_datasize, test_size=0.2, stratify=y_datasize, random_state=seed
    )

    # 検証データを全体の20%にする
    X_train_datasize, X_val_datasize, y_train_datasize, y_val_datasize = train_test_split(
        X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=seed
    )

    # 学習データの一部を分割する
    # 学習データが全体の60％なので、(percentage_split/60)を指定することで、全体のpercentage_splitにする

    _, X_train_splitted, _, y_train_splitted = train_test_split(
        X_train_datasize, y_train_datasize,
        test_size=(percentage_split/60), 
        stratify=y_train_datasize, 
        random_state=seed
    )

    sample_nums_stratified = 30 # 特徴量の数は30個に固定
    dfs = [] # classification reportの一時的な格納リスト

    # ループの50回で数十分から100分くらいかかるので、ローカルでやる場合は50を仮の上限にする。
    for i in range(50):

        clear_output(wait=True) # forループのなかで、現在の回だけを表示する

        print(f"percentage_split:{percentage_split}, {i}回目")

        # 層化して、ランダムでカラム名を抽出する

        non_binary_col_names = X_train_datasize[list_non_binary].sample(
            n=dict_sample_nums[sample_nums_stratified]["non_binary"],
            axis=1,
            random_state=i # ループごとにseedを毎回変えることで、毎回ランダムなカラムが抽出される+range()なので、再現性もある
        ).columns.tolist()

        binary_col_names = X_train_datasize.drop(list_non_binary, axis=1).sample(
            n=dict_sample_nums[sample_nums_stratified]["binary"],
            axis=1,
            random_state=i # ループごとにseedを毎回変えることで、毎回ランダムなカラムが抽出される+range()なので、再現性もある
        ).columns.tolist()

        # binaryカラムと非binaryカラムを統合して、学習に使うカラムリストを作成
        feature_cols = non_binary_col_names + binary_col_names

        # 抽出したカラム名で学習、予測を行う
        dict_report_stratified_datasize = train_predict_eval(
            X_train=X_train_splitted[feature_cols],
            y_train=y_train_splitted, 
            X_val=X_val_datasize[feature_cols],
            y_val=y_val_datasize, 
            X_test=X_test_datasize[feature_cols],
            y_test=y_test_datasize
        )

        # 転値して、行=クラス名、列=指標 にする
        df_report_stratified_datasize = pd.DataFrame(dict_report_stratified_datasize).T
        
        dfs.append(df_report_stratified_datasize)

    # 結果を格納したDataFrameの同じ位置のセル同士で平均をとる
    df_mean_report_stratified_datasize = pd.concat(dfs).groupby(level=0).mean()

    # 平均を取る前の元データも保存しておく
    pd.concat(dfs).to_csv(
        f"../outputs/{percentage_split}_stratified_col{sample_nums_stratified}_raw.csv",
        index=True, # indexがクラスなので、クラスをのこす
        encoding="utf-8"
    )

    # 平均の結果も保存しておく
    df_mean_report_stratified_datasize.to_csv(
        f"../outputs/{percentage_split}_stratified_col{sample_nums_stratified}_mean.csv",
        index=True, # indexがクラスなので、クラスをのこす
        encoding="utf-8"
    )

In [ ]:

for i in [0.5]:
# for i in [1,5,10,20,30,40,50]:


    tmp_night_calc(percentage_split=i)

### 4.3 95%信頼区間の確認

In [ ]:
col_names =  [
    "precision_mean", "precision_std", "precision_ucl", "precision_lcl",
    "recall_mean","recall_std", "recall_ucl", "recall_lcl", 
    "f1-score_mean","f1-score_std" ,"f1-score_ucl", "f1-score_lcl"
]

df_result = pd.DataFrame(columns=col_names)

for i in [0.5, 1, 5, 10, 20, 30, 40, 50]:

    df = pd.read_csv(f"../outputs/{i}_stratified_col30_raw.csv")

    # cover_typeごとにforを回す
    for cover_type in range(1,8):

        # 計算結果を一時的に保存
        list_tmp =[]

        for col in ["precision", "recall", "f1-score"]:
            avg = df[col][df[df.columns[0]]==str(cover_type)].mean()
            std = df[col][df[df.columns[0]]==str(cover_type)].std()
            sample_volume = len(df[col][df[df.columns[0]]==str(cover_type)])

            # 95%信頼区間の上限
            ucl = avg + 1.96*(
                    std/(sample_volume**0.5)
                )

            # 95%信頼区間の下限
            lcl = avg - 1.96*(
                    std/(sample_volume**0.5)
                )

            list_tmp.extend([avg, std, ucl, lcl])
        
        df_result.loc[f"{i}_{cover_type}", col_names] = list_tmp

In [ ]:
df_result